In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q transformers  # Thư viện Hugging Face để tải mô hình Gemma
!pip install -q accelerate    # Hỗ trợ chạy mô hình trên GPU
!pip install -q evaluate      # Thư viện tính các chỉ số đánh giá
!pip install -q rouge-score
!pip install -q bert-score
!pip install -q sentencepiece # Tokenizer của Gemma sử dụng SentencePiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.1 MB/s eta 0:00:00


Khai báo đường dẫn

In [3]:
import os

PROJECT_DIR = "/content/drive/MyDrive/gemmapoem"          # Thư mục chứa toàn bộ dự án
MODEL_PATH = f"{PROJECT_DIR}/models/gemma2b-lucbat"       # Thư mục chứa mô hình Gemma đã fine-tune
TEST_FILE = f"{PROJECT_DIR}/data/test.txt"                # File dữ liệu kiểm thử
RESULT_DIR = f"{PROJECT_DIR}/results"                     # Thư mục lưu kết quả đánh giá
os.makedirs(RESULT_DIR, exist_ok=True)                    # Nếu chưa có thư mục results thì tạo mới

# Kiểm tra đường dẫn
print("Project:", PROJECT_DIR)
print("Model :", MODEL_PATH)
print("Test  :", TEST_FILE)
print("Result:", RESULT_DIR)

Project: /content/drive/MyDrive/gemmapoem
Model : /content/drive/MyDrive/gemmapoem/models/gemma2b-lucbat
Test  : /content/drive/MyDrive/gemmapoem/data/test.txt
Result: /content/drive/MyDrive/gemmapoem/results


Load mô hình gemma22b

In [4]:
import transformers
import torch
from transformers import (        # Thư viện Hugging Face
    AutoTokenizer,
    AutoModelForCausalLM
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.eval()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

Gemma2ForCausalLM(
  (model): Gemma2Model(
    (embed_tokens): Gemma2TextScaledWordEmbedding(256000, 2304, padding_idx=0)
    (layers): ModuleList(
      (0-25): 26 x Gemma2DecoderLayer(
        (self_attn): Gemma2Attention(
          (q_proj): Linear(in_features=2304, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2304, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2304, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2304, bias=False)
        )
        (mlp): Gemma2MLP(
          (gate_proj): Linear(in_features=2304, out_features=9216, bias=False)
          (up_proj): Linear(in_features=2304, out_features=9216, bias=False)
          (down_proj): Linear(in_features=9216, out_features=2304, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (post_attention_layernorm): Gemma2RMSNorm((2304,), eps=1e-06)
        (pre_feedforward_lay

Đọc dữ liệu kiểm thử

In [5]:
import os         # Làm việc với thư mục và tập tin

# Kiểm tra file test
if not os.path.exists(TEST_FILE):
    raise FileNotFoundError(f"Không tìm thấy file:\n{TEST_FILE}")

# Đọc toàn bộ các dòng
with open(TEST_FILE, "r", encoding="utf-8") as f:
    lines = [line.strip() for line in f if line.strip()]

# Tạo tập kiểm thử
test_data = []
i = 0
while i < len(lines) - 1:

    cau_luc = lines[i]
    cau_bat = lines[i + 1]

    # Chỉ lấy đúng cặp lục (6 tiếng) - bát (8 tiếng)
    if len(cau_luc.split()) == 6 and len(cau_bat.split()) == 8:
        test_data.append((cau_luc, cau_bat))
        i += 2
    else:
        i += 1

# Cắt lấy 10% tập dữ liệu
limit = max(1, int(len(test_data) * 0.1))
test_data = test_data[:limit]

print(f"Tổng số mẫu kiểm thử: {len(test_data)}")
# Hiển thị ví dụ
if len(test_data) > 0:
    print("\nVí dụ:")
    print("Câu lục :", test_data[0][0])
    print("Câu bát :", test_data[0][1])

Tổng số mẫu kiểm thử: 28118

Ví dụ:
Câu lục : tiếc thay duyên tấn phận tần
Câu bát : chưa quen đã lạ chưa gần đã xa


Hàm đánh giá luật thơ và khởi tạo PhoBERTScore

In [6]:
import time
import unicodedata
from bert_score import BERTScorer

# Khởi tạo PhoBERTScore
print("Đang khởi tạo PhoBERTScore...")
scorer = BERTScorer(
    model_type="vinai/phobert-base",
    num_layers=9,
    rescale_with_baseline=False
)

# Các hàm đánh giá luật thơ
TRAC_CHARS = set(
    "áắấéếíóốớúứý"
    "ảẳẩẻểỉỏổởủửỷ"
    "ãẵẫẽễĩõỗỡũữỹ"
    "ạặậẹệịọộợụựỵ"
)

def get_tone(word):
    word = word.lower()
    if not any(c.isalpha() for c in word):
        return None
    return "T" if any(c in TRAC_CHARS for c in word) else "B"

def remove_tones(word):
    text = unicodedata.normalize("NFD", word)
    return "".join(
        c for c in text
        if unicodedata.category(c) != "Mn"
    )

def get_rhyme_part(word):
    word = remove_tones(word.lower().strip())
    consonants = [
        "ngh","ch","gh","gi","kh",
        "ng","nh","ph","qu","th","tr",
        "b","c","d","đ","g","h",
        "k","l","m","n","p","q",
        "r","s","t","v","x"
    ]

    for c in consonants:
        if word.startswith(c):
            return word[len(c):]
    return word

def is_rhyme(word1, word2):
    if not word1 or not word2:
        return False
    return get_rhyme_part(word1) == get_rhyme_part(word2)

def evaluate_luc_bat_rules(cau_luc, cau_bat_gen):
    luc_words = cau_luc.strip().split()
    bat_words = cau_bat_gen.strip().split()

    score_length = 0.0
    score_tone = 0.0
    score_rhyme = 0.0

    # Độ dài
    if len(bat_words) == 8:
        score_length = 10.0

    # Thanh điệu
    if len(bat_words) >= 8:
        tone_score = 30.0 / 4.0

        if get_tone(bat_words[1]) == "B":
            score_tone += tone_score

        if get_tone(bat_words[3]) == "T":
            score_tone += tone_score

        if get_tone(bat_words[5]) == "B":
            score_tone += tone_score

        if get_tone(bat_words[7]) == "B":
            score_tone += tone_score

    # Gieo vần
    if len(luc_words) >= 6 and len(bat_words) >= 6:
        if is_rhyme(luc_words[5], bat_words[5]):
            score_rhyme = 60.0
    total_score = score_length + score_tone + score_rhyme

    return (
        total_score,
        score_length,
        score_tone,
        score_rhyme
    )

Đang khởi tạo PhoBERTScore...


config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/895k [00:00<?, ?B/s]

bpe.codes:   0%|          | 0.00/1.14M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.13M [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  543MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: vinai/phobert-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.decoder.weight    | UNEXPECTED |  | 
lm_head.decoder.bias      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
import torch
from transformers import LogitsProcessor

class StrictLucBatProcessor(LogitsProcessor):
    def __init__(self, tokenizer, max_total_lines=2):
        self.tokenizer = tokenizer
        self.trac_chars = set("áắấéếíóốớúứýảẳẩẻểỉỏổởủửỷãẵẫẽễĩõỗỡũữỹạặậẹệịọộợụựỵ")
        self.vowels = set("aeiouyáàảãạăắằẳẵặâấầẩẫậéèẻẽẹêếềểễệíìỉĩịóòỏõọôốồổỗộơớờởỡợúùủũụưứừửữựýỳỷỹỵ")
        self.nl_token_id = tokenizer.encode('\n', add_special_tokens=False)[-1]
        self.eos_token_id = tokenizer.eos_token_id
        self.max_total_lines = max_total_lines

    def get_tone(self, text):
        text = text.lower()
        if not any(c in self.vowels for c in text): return None
        if any(c in self.trac_chars for c in text): return "T"
        return "B"

    def remove_tones(self, word):
        s1 = u'ÀÁÂÃÈÉÊÌÍÒÓÔÕÙÚÝàáâãèéêìíòóôõùúýĂăĐđĨĩŨũƠơƯưẠạẢảẤấẦầẨẩẪẫẬậẮắẰằẲẳẴẵẶặẸẹẺẻẼẽẾếỀềỂểỄễỆệỈỉỊịỌọỎỏỐốỒồỔổỖỗỘộỚớỜờỞởỠỡỢợỤụỦủỨứỪừỬửỮữỰựỲỳỴỵỶỷỸỹ'
        s0 = u'AAAAEEEIIOOOOUUYaaaaeeeiioooouuyAaDdIiUuOoUuAaAaAaAaAaAaAaAaAaAaAaAaEeEeEeEeEeEeEeEeIiIiOoOoOoOoOoOoOoOoOoOoOoOoUuUuUuUuUuUuUuYyYyYyYy'
        s = ''
        for c in word:
            if c in s1: s += s0[s1.index(c)]
            else: s += c
        return s

    def get_rhyme_part(self, word):
        word = self.remove_tones(word.lower().strip())
        consonants = ['ngh', 'ch', 'gh', 'gi', 'kh', 'ng', 'nh', 'ph', 'qu', 'th', 'tr',
                      'b', 'c', 'd', 'đ', 'g', 'h', 'k', 'l', 'm', 'n', 'p', 'q', 'r', 's', 't', 'v', 'x']
        for c in consonants:
            if word.startswith(c): return word[len(c):]
        return word

    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor) -> torch.FloatTensor:
        text = self.tokenizer.decode(input_ids[0])
        lines = text.split('\n')
        current_line = lines[-1]

        previous_lines = [l.strip() for l in lines[:-1] if l.strip()]
        completed_lines = len(previous_lines)

        if not previous_lines: target_length = 6
        else: target_length = 8 if len(previous_lines[-1].split()) == 6 else 6

        current_words = current_line.strip().split()
        current_word_count = len(current_words)
        is_finishing_poem = (completed_lines >= self.max_total_lines - 1) and (target_length == 8)

        top_k_val, top_indices = torch.topk(scores[0], 100)
        mask = torch.full_like(scores[0], -float('inf'))

        for token_id in top_indices:
            token_str = self.tokenizer.decode([token_id])
            is_valid = True

            candidate_line = current_line + token_str
            candidate_words = candidate_line.strip().split()
            candidate_word_count = len(candidate_words)

            # 1. LUẬT ĐỘ DÀI
            if '\n' in token_str or token_id == self.eos_token_id:
                if current_word_count < target_length: is_valid = False
                elif current_word_count == target_length:
                    if current_words and not any(c in self.vowels for c in current_words[-1].lower()):
                        is_valid = False
                    if is_valid:
                        if is_finishing_poem and '\n' in token_str: is_valid = False
                        elif not is_finishing_poem and token_id == self.eos_token_id: is_valid = False
            else:
                if candidate_word_count > target_length: is_valid = False

                # 2. LUẬT THANH ĐIỆU
                if is_valid and candidate_word_count in [2, 4, 6, 8]:
                    target_tone = "B" if candidate_word_count in [2, 6, 8] else "T"
                    tone = self.get_tone(candidate_words[-1])
                    if tone and tone != target_tone: is_valid = False

                # 3. LUẬT GIEO VẦN (MỚI THÊM)
                # Chỉ ép vần khi đang gõ chữ thứ 6 của câu Bát
                if is_valid and candidate_word_count == 6 and target_length == 8 and completed_lines > 0:
                    luc_words = previous_lines[-1].split()
                    if len(luc_words) == 6:
                        target_rhyme = self.get_rhyme_part(luc_words[5])
                        current_word = candidate_words[-1]

                        # Chỉ check vần khi từ này ĐÃ CÓ NGUYÊN ÂM (không chém nhầm phụ âm dở dang)
                        if any(c in self.vowels for c in current_word.lower()):
                            current_rhyme = self.get_rhyme_part(current_word)
                            if current_rhyme and target_rhyme:
                                # Cho phép nếu vần hiện tại là tiền tố của vần đích (đang gõ dở token)
                                if not target_rhyme.startswith(current_rhyme):
                                    is_valid = False

            if is_valid: mask[token_id] = scores[0, token_id]

        scores[0] = mask

        # ĐIỀU HƯỚNG KẾT THÚC
        if current_word_count == target_length:
            if current_words and any(c in self.vowels for c in current_words[-1].lower()):
                if is_finishing_poem: scores[0, self.eos_token_id] += 50.0
                else: scores[0, self.nl_token_id] += 25.0

        if torch.all(scores[0] == -float('inf')):
            if current_word_count >= target_length:
                if is_finishing_poem: scores[0, self.eos_token_id] = 10.0
                else: scores[0, self.nl_token_id] = 10.0
            else: scores[0, self.eos_token_id] = 10.0

        return scores

Sinh thơ và đánh giá

In [8]:
import os
import json
import time
import evaluate

RESULT_FILE = os.path.join(
    RESULT_DIR,
    "evaluation_results.jsonl"
)

CHECKPOINT_FILE = os.path.join(
    RESULT_DIR,
    "evaluation_checkpoint.json"
)

generated_bats = []
reference_bats = []

total_rule = 0
total_len = 0
total_tone = 0
total_rhyme = 0
start_idx = 0

# Đọc CHECKPOINT

if os.path.exists(CHECKPOINT_FILE):

    print("Đã tìm thấy checkpoint.")

    with open(
        CHECKPOINT_FILE,
        "r",
        encoding="utf-8"
    ) as f:

        checkpoint = json.load(f)

    start_idx = checkpoint["current_index"]

    total_rule = checkpoint["total_rule"]
    total_len = checkpoint["total_len"]
    total_tone = checkpoint["total_tone"]
    total_rhyme = checkpoint["total_rhyme"]

    generated_bats = checkpoint["generated_bats"]
    reference_bats = checkpoint["reference_bats"]

    print(f"Tiếp tục từ mẫu {start_idx}")

else:
    print("Không có checkpoint. Đánh giá từ đầu.")
    if os.path.exists(RESULT_FILE):
        os.remove(RESULT_FILE)

start_time = time.time()
print(f"\nBắt đầu đánh giá từ mẫu {start_idx}/{len(test_data)}\n")

# Sinh thơ và đánh giá

from transformers import LogitsProcessorList
processor = StrictLucBatProcessor(
    tokenizer,
    max_total_lines=2
)
logits_processor_list = LogitsProcessorList([processor])
for idx in range(start_idx, len(test_data)):
    cau_luc, cau_bat_ref = test_data[idx]
    prompt = cau_luc + "\n"
    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=25,
            logits_processor=logits_processor_list,
            do_sample=True,
            temperature=0.8,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    # Lấy câu bát sinh ra

    full_text = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )
    cau_bat_gen = (
        full_text
        .replace(cau_luc, "")
        .strip()
        .split("\n")[0]
        .strip()
    )

    # Đánh giá luật

    rule_score, len_score, tone_score, rhyme_score = \
        evaluate_luc_bat_rules(
            cau_luc,
            cau_bat_gen
        )

    total_rule += rule_score
    total_len += len_score
    total_tone += tone_score
    total_rhyme += rhyme_score

    generated_bats.append(cau_bat_gen)
    reference_bats.append(cau_bat_ref)

    # Ghi từng mẫu

    sample = {
        "index": idx,
        "cau_luc": cau_luc,
        "reference": cau_bat_ref,
        "generated": cau_bat_gen,
        "rule_score": rule_score,
        "length_score": len_score,
        "tone_score": tone_score,
        "rhyme_score": rhyme_score
    }
    with open(
        RESULT_FILE,
        "a",
        encoding="utf-8"
    ) as f:
        json.dump(
            sample,
            f,
            ensure_ascii=False
        )
        f.write("\n")


    # Lưu checkpoint

    if (idx + 1) % 10 == 0 or (idx + 1) == len(test_data):
        checkpoint = {
            "current_index": idx + 1,
            "total_rule": total_rule,
            "total_len": total_len,
            "total_tone": total_tone,
            "total_rhyme": total_rhyme,
            "generated_bats": generated_bats,
            "reference_bats": reference_bats
        }
        with open(
            CHECKPOINT_FILE,
            "w",
            encoding="utf-8"
        ) as f:
            json.dump(
                checkpoint,
                f,
                ensure_ascii=False
            )
        print(f"Đã xử lý [{idx+1}/{len(test_data)}] mẫu - Đã lưu checkpoint.")


# Tính thời gian

eval_time = time.time() - start_time
print("\nĐã sinh xong toàn bộ câu bát.")

Đã tìm thấy checkpoint.
Tiếp tục từ mẫu 27110

Bắt đầu đánh giá từ mẫu 27110/28118

Đã xử lý [27120/28118] mẫu - Đã lưu checkpoint.
Đã xử lý [27130/28118] mẫu - Đã lưu checkpoint.
Đã xử lý [27140/28118] mẫu - Đã lưu checkpoint.
Đã xử lý [27150/28118] mẫu - Đã lưu checkpoint.
Đã xử lý [27160/28118] mẫu - Đã lưu checkpoint.
Đã xử lý [27170/28118] mẫu - Đã lưu checkpoint.
Đã xử lý [27180/28118] mẫu - Đã lưu checkpoint.
Đã xử lý [27190/28118] mẫu - Đã lưu checkpoint.
Đã xử lý [27200/28118] mẫu - Đã lưu checkpoint.
Đã xử lý [27210/28118] mẫu - Đã lưu checkpoint.
Đã xử lý [27220/28118] mẫu - Đã lưu checkpoint.
Đã xử lý [27230/28118] mẫu - Đã lưu checkpoint.
Đã xử lý [27240/28118] mẫu - Đã lưu checkpoint.
Đã xử lý [27250/28118] mẫu - Đã lưu checkpoint.
Đã xử lý [27260/28118] mẫu - Đã lưu checkpoint.
Đã xử lý [27270/28118] mẫu - Đã lưu checkpoint.
Đã xử lý [27280/28118] mẫu - Đã lưu checkpoint.
Đã xử lý [27290/28118] mẫu - Đã lưu checkpoint.
Đã xử lý [27300/28118] mẫu - Đã lưu checkpoint.
Đã x

Tính PhoBERTScore


In [9]:
print("\nĐang tính PhoBERTScore...")

P, R, F1 = scorer.score(
    generated_bats,
    reference_bats
)

avg_phobert = F1.mean().item() * 100

eval_time = time.time() - start_time

n = len(test_data)


print("\nĐánh giá hoàn tất.")


Đang tính PhoBERTScore...

Đánh giá hoàn tất.


Lưu kết quả và báo cáo đánh giá

In [10]:
import os
import json
import shutil                  # Nén thư mục kết quả
SAVE_DIR = RESULT_DIR          # dùng cùng thư mục đã lưu ở Cell 8
os.makedirs(SAVE_DIR, exist_ok=True)

# File kết quả chi tiết
detail_file = RESULT_FILE


# Kết quả tổng hợp
summary = {
    "num_samples": n,
    "rule_score": round(total_rule / n, 2),
    "length_score": round(total_len / n, 2),
    "tone_score": round(total_tone / n, 2),
    "rhyme_score": round(total_rhyme / n, 2),
    "phobert_score": round(avg_phobert, 2),
    "evaluation_time_seconds": round(eval_time, 2),
    # "valid_poems_percent": round(valid_count / n * 100, 2)
}

summary_file = os.path.join(
    SAVE_DIR,
    "evaluation_summary.json"
)

with open(summary_file, "w", encoding="utf-8") as f:
    json.dump(
        summary,
        f,
        ensure_ascii=False,
        indent=4
    )

Báo cáo đánh giá

In [14]:
print("\n")
print("=" * 70)
print("                  BÁO CÁO ĐÁNH GIÁ MÔ HÌNH GEMMA-2B")
print("=" * 70)

minutes = eval_time / 60

print(f"Tổng số mẫu test : {n}")
print(f"Thời gian chạy   : {eval_time:.1f} giây (~{minutes:.1f} phút)")

print()

print("1. ĐÁNH GIÁ LUẬT THƠ (Rule-based)")
print(f"   - Điểm tổng : {total_rule/n:.2f}/100")
print(f"      + Độ dài (10%)      : {total_len/n:.2f}")
print(f"      + Thanh điệu (30%)  : {total_tone/n:.2f}")
print(f"      + Gieo vần (60%)    : {total_rhyme/n:.2f}")

print()

print("2. PHOBERT SCORE")
print(f"   - F1 Score : {avg_phobert:.2f}")

print()

print("=" * 70)
print("ĐÃ LƯU KẾT QUẢ")
print("=" * 70)
print(f"Chi tiết : {detail_file}")
print(f"Tổng hợp : {summary_file}")
print("=" * 70)



                  BÁO CÁO ĐÁNH GIÁ MÔ HÌNH GEMMA-2B
Tổng số mẫu test : 28118
Thời gian chạy   : 1638.8 giây (~27.3 phút)

1. ĐÁNH GIÁ LUẬT THƠ (Rule-based)
   - Điểm tổng : 91.66/100
      + Độ dài (10%)      : 9.99
      + Thanh điệu (30%)  : 29.93
      + Gieo vần (60%)    : 51.74

2. PHOBERT SCORE
   - F1 Score : 39.73

ĐÃ LƯU KẾT QUẢ
Chi tiết : /content/drive/MyDrive/gemmapoem/results/evaluation_results.jsonl
Tổng hợp : /content/drive/MyDrive/gemmapoem/results/evaluation_summary.json
